# SignVLM: batch video augmentation (Colab)

Generates up to **9** augmented `*.mp4` files per **label folder** in `validation_data`, `test_data`, and `unseen_data` only (not `train_data`), in **place** (new files in the same class subfolders; no separate output root).

Uses `augmentation_combined.py` (OpenCV `mp4v`). A **class folder** is **skipped** if it already has any `*_aug_*.mp4`. Remaining classes are run **in parallel** (`ProcessPoolExecutor`); set **`AUG_MAX_WORKERS`** (e.g. `2` or `4`) to cap processes if you hit OOM. With 9 per label, new files are `{label}_aug_16` … `aug_24` (existing target names are skipped inside the run).

**Env overrides (optional):** `SIGNVLM_DRIVE_BASE` (default on Colab: `.../MyDrive/FYP_DATA`); `SIGNVLM_REPO_ROOT` (repo path; must contain `augmentation_and_preprocessing/augmentation/`); else `{BASE}/Urdu-Sign-Language-Recognition-using-SignVLM` or `SIGNVLM_REPO_SUBDIR`. Off Colab, `SIGNVLM_BASE` defaults to `./signvlm_data` for `BASE`.

In-place: no new dataset directories; a missing split folder is skipped with one line.

In [ ]:
# Installs: OpenCV (video read/write) + NumPy; works with Python 3 on Colab and locally.
import subprocess
import sys

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install(["opencv-python-headless", "numpy"])
print("OK: opencv-python-headless, numpy")

In [ ]:
# Mount Drive (Colab) and set paths — see markdown above for env vars.
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MY = "/content/drive/MyDrive"
else:
    DRIVE_MY = os.environ.get("SIGNVLM_DRIVE_MY", "")

BASE = os.environ.get("SIGNVLM_DRIVE_BASE", "").strip()
if not BASE:
    if IN_COLAB and DRIVE_MY:
        BASE = f"{DRIVE_MY}/FYP_DATA"
    else:
        BASE = os.environ.get("SIGNVLM_BASE", os.path.join(os.getcwd(), "signvlm_data"))

REPO_SUBDIR = os.environ.get("SIGNVLM_REPO_SUBDIR", "Urdu-Sign-Language-Recognition-using-SignVLM").strip()
REPO_ROOT = os.environ.get("SIGNVLM_REPO_ROOT", "").strip()
if not REPO_ROOT:
    REPO_ROOT = str(Path(BASE) / REPO_SUBDIR)
REPO_ROOT = str(Path(REPO_ROOT).resolve())
AUG_DIR = Path(REPO_ROOT) / "augmentation_and_preprocessing" / "augmentation"
print(f"IN_COLAB={IN_COLAB} BASE={BASE}\nREPO_ROOT={REPO_ROOT}\nAUG_DIR={AUG_DIR}")
if not Path(BASE).is_dir():
    print("WARNING: BASE is not a directory. Set SIGNVLM_DRIVE_BASE or signvlm_data layout.")
if not AUG_DIR.is_dir():
    print(f"ERROR: Augmentation code not found at: {AUG_DIR}. Set SIGNVLM_REPO_ROOT.")

In [ ]:
# Import augmentation (adds same folder that holds augmentation_combined.py).
import os
import sys
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

aug = Path(REPO_ROOT) / "augmentation_and_preprocessing" / "augmentation"
# Spawned ProcessPool workers must resolve this package (extend PYTHONPATH, not just sys.path in the notebook).
_augp = str(aug)
if _augp not in os.environ.get("PYTHONPATH", ""):
    os.environ["PYTHONPATH"] = _augp + (
        os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
    )
if _augp not in sys.path:
    sys.path.insert(0, _augp)
from augmentation_combined import (  # noqa: E402
    augment_one_label_entry,
    label_dir_has_aug_videos,
)

In [ ]:
# Per-label: skip if the class folder already has any *_aug_*.mp4; else augment in parallel.
from pathlib import Path

SPLITS = ("validation_data", "test_data", "unseen_data")
AUGMENTED_PER_LABEL = 9
MAX_WORKERS = max(1, int(os.environ.get("AUG_MAX_WORKERS", "4")))


def collect_jobs():
    jobs = []
    skipped_aug = []
    for split_name in SPLITS:
        root = Path(BASE) / split_name
        if not root.is_dir():
            print(f"skip (not found): {root}")
            continue
        for label_dir in sorted(d for d in root.iterdir() if d.is_dir()):
            if label_dir_has_aug_videos(label_dir):
                skipped_aug.append(f"{split_name}/{label_dir.name}")
                continue
            jobs.append((str(label_dir), AUGMENTED_PER_LABEL, True))
    return jobs, skipped_aug


jobs, skipped_aug = collect_jobs()
if skipped_aug:
    print(f"Skipped {len(skipped_aug)} label(s) (already have *_aug_*.mp4):")
    for s in skipped_aug:
        print("  -", s)
if not jobs:
    print("Nothing to do: all present labels have aug, or no label folders found.")
else:
    n_workers = min(MAX_WORKERS, len(jobs))
    print(
        f"Running {len(jobs)} label(s) with {n_workers} process(es) "
        "(set AUG_MAX_WORKERS; worker logs may interleave)."
    )
    total_s, total_f = 0, 0
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        for s, f, _ in ex.map(augment_one_label_entry, jobs, chunksize=1):
            total_s += s
            total_f += f
    print(f"Done. total success: {total_s}, total failed: {total_f}")